# 04 · 지표·분모·관찰 구간

목표: 누적 counter와 구간 비율을 구분하고 histogram을 읽는다. 약15분.

**실행:** Python 3.11 공통 .venv 커널에서 위→아래. 다른 Notebook·서버·Docker·모델 설치 없이 독립 실행. 코드는 MOCK 고정 문장·대기만 사용한다. 실제 LLM 품질·처리량 측정이 아니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from week10.lab.experiments import lab_client, window_delta

## 관찰 전 스냅샷

상태 확인·UI·metrics 조회는 assist 분모에서 제외한다.

In [ ]:
async with lab_client() as (app, client):
    await client.get("/health/live")
    await client.get("/metrics")
    before = app.state.telemetry.snapshot()
    for scenario in ["normal", "normal", "retrieval_error", "provider_error"]:
        await client.post("/api/v1/assist", json={"message": "예시", "scenario": scenario})
    await client.post("/api/v1/assist", json={"message": " "})
    after = app.state.telemetry.snapshot()
    exposition = (await client.get("/metrics")).text
assert before["total"] == 0 and after["total"] == 5

## 기간 차이

5xx/전체 assist 요청으로 2/5를 계산한다. 422 포함 여부를 먼저 명시한다.

In [ ]:
delta = window_delta(before, after)
assert delta["requests"] == 5 and delta["5xx"] == 2
assert delta["error_rate"] == 0.4
delta

## 텍스트 노출 형식

counter·gauge·histogram이 어떤 줄인지 찾는다. Prometheus 서버는 띄우지 않는다.

In [ ]:
print(exposition)
assert "trace_id" not in exposition and "request_id" not in exposition

## 누적 버킷

le=0.1은 0.1초 이하 전체다. 버킷을 단순 합산하면 중복된다.

In [ ]:
buckets = [line for line in exposition.splitlines() if "_bucket" in line]
counts = [float(line.split()[-1]) for line in buckets]
assert counts == sorted(counts) and counts[-1] == 5
buckets

## 성공 지연과 전체 지연

실패가 빨리 반환되면 전체 평균이 작아질 수 있다. logs는 최근 200건만이다.

In [ ]:
success = [r["duration_ms"] for r in after["logs"] if r["status"] == 200]
all_ms = [r["duration_ms"] for r in after["logs"]]
print("성공 평균:", sum(success) / len(success), "전체 평균:", sum(all_ms) / len(all_ms))
print("분모:", len(success), len(all_ms))

## 트래픽 없는 구간

요청0은 100% 성공이 아니라 관측 불충분이다.

In [ ]:
assert window_delta(after, after)["error_rate"] is None
try:
    window_delta(after, before)
except ValueError as e:
    print("재시작/역순 거부:", e)

## 알림 판단

수업용 규칙. 운영 SLO·확정 배점이 아니다.

In [ ]:
def review_signal(d):
    return (
        "관측 부족" if d["requests"] < 5 else "원인 확인" if d["error_rate"] > 0.1 else "관찰 유지"
    )


print(review_signal(delta))
print(review_signal(window_delta(after, after)))

## 프로젝트 적용 기록

예상 → 관찰 → 근거 → 다음 변경을 한 줄씩 기록한다. 실제로 실행하지 않은 조건은 미측정으로 표시한다. 학생 정보나 원문 입력은 공개 저장소에 올리지 않는다.